# QuditQuantumCircuit — Feature Tour

`QuditQuantumCircuit` is the qudit analogue of Qiskit's `QuantumCircuit`. It **owns** an encoded `QuantumCircuit` (`.circuit`) where every `d`-level qudit is represented by `ceil(log2(d))` qubits, so the result can be transpiled and run on any Qiskit backend/primitive.

This notebook is a guided tour of the public API:

- building circuits (integer shorthand vs. explicit registers, mixed dimensions),
- addressing qudits/cldigits (index, list, slice, register, `Qudit` object),
- the single-qudit gate library and controlled/multi-qudit gates,
- state preparation (`initialize_levels`, `initialize`),
- directives (`barrier`, `reset`, `measure`, `measure_all`),
- the *ideal* vs *real* vs *decomposed* circuit views,
- structural operations (`copy`, `compose`, `inverse`),
- circuit metrics (`size`, `depth`, `width`, `count_ops`),
- decoding measurement results back into qudit levels,
- and the `QuditCircuitError` guardrails.

Every section ends with one or more `assert`s that double as a live correctness check of the library.

**Conventions** (identical to Qiskit): everything is little-endian. Qubit `j` of a qudit carries weight `2**j`. In an `initialize_levels` level-string, the *rightmost* token is the *first* (lowest-index) target qudit. In a decoded counts key, cldigit 0 comes first.

In [ ]:
# NOTE: qiskit-aer is not a requirement of this library
# but it's required to fully run this notebook
# ! pip install qiskit-aer>=qiskit-aer-0.17.2

In [2]:
# For the sake of running the notebook add the library path
import sys
sys.path.append("./src")

In [3]:
import numpy as np

from qiskit import transpile
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity
from qiskit_aer import AerSimulator

from qiskit_qudits.circuit import QuditQuantumCircuit
from qiskit_qudits.circuit.qudit import QuditRegister
from qiskit_qudits.circuit.cldigit import ClDigitRegister
from qiskit_qudits.circuit.exceptions import QuditCircuitError
from qiskit_qudits.utils.encoding import format_levels

np.set_printoptions(precision=3, suppress=True)
print("qiskit-qudits example environment ready.")

qiskit-qudits example environment ready.


## Setup: a tiny sampling helper

We use `AerSimulator` for shot-based sampling and `qc.decode_counts(...)` to turn raw bit-strings back into qudit levels. `transpile` is called exactly as the class docstring recommends: `transpile(qudit_circuit.circuit, backend)`.

In [4]:
backend = AerSimulator()


def sample_counts(qqc, shots=8000, seed=42):
    """Run a QuditQuantumCircuit on AerSimulator and decode the outcome.

    Returns the raw (bit-string) counts together with the decoded
    ``{levels: shots}`` mapping produced by ``qqc.decode_counts``.
    """
    transpiled = transpile(qqc.circuit, backend)
    job = backend.run(transpiled, shots=shots, seed_simulator=seed)
    raw_counts = job.result().get_counts()
    return raw_counts, qqc.decode_counts(raw_counts)


def show_decoded(decoded, total=None):
    """Pretty-print a decoded counts mapping, sorted by levels."""
    total = total or sum(decoded.values())
    for levels, shots in sorted(decoded.items()):
        print(f"  levels={format_levels(levels)!s:<10} shots={shots:5d}  ({shots / total:6.2%})")

## 1. Building circuits

Two constructor styles are supported:

- the **integer shorthand**, `QuditQuantumCircuit(n_qudits, n_cldigits, dim=d)`, which auto-creates a `'qd'` qudit register and a `'cb'` cldigit register;
- explicit **registers**, `QuditQuantumCircuit(QuditRegister(...), ClDigitRegister(...))`, needed as soon as a circuit mixes dimensions.

In [5]:
# --- Integer shorthand: 2 qutrits + 2 cldigits sized for a qutrit outcome ---
qc = QuditQuantumCircuit(2, 2, dim=3, name="basics")

assert qc.num_qudits == 2
assert qc.num_cldigits == 2
assert qc.dims == (3, 3)
assert qc.dim == 3                 # homogeneous circuit -> single dim available
assert qc.num_qubits == 4          # each qutrit needs ceil(log2(3)) = 2 qubits
assert qc.cldigit_dims == (3, 3)
assert qc.cldigit_widths == (2, 2)  # each cldigit needs 2 clbits to hold a qutrit outcome

print(repr(qc))

<QuditQuantumCircuit 'basics': 2 qudit(s) dims=(3, 3), 2 cldigit(s), 0 instruction(s), 4 qubit(s)>


In [6]:
# --- Explicit registers: mixing dimensions in a single circuit ---
alice = QuditRegister(2, 3, "alice")           # 2 qutrits
bob = QuditRegister(1, 4, "bob")               # 1 ququart
outcomes = ClDigitRegister.from_dims((3, 3, 4), "cb")

mixed = QuditQuantumCircuit(alice, bob, outcomes)

assert mixed.num_qudits == 3
assert mixed.dims == (3, 3, 4)
assert mixed.qdregs == (alice, bob)
assert mixed.has_register(alice) and mixed.has_register(bob)

# `.dim` only makes sense for a homogeneous circuit:
try:
    mixed.dim
except QuditCircuitError as exc:
    print("Expected error for a heterogeneous circuit:", exc)

Expected error for a heterogeneous circuit: 'this circuit is heterogeneous ([3, 4]); use `dims` instead of `dim`.'


In [7]:
# --- Metadata, naming and global phase are first-class citizens ---
annotated = QuditQuantumCircuit(
    1, dim=3, name="annotated", global_phase=np.pi / 4, metadata={"experiment": "demo"},
)
assert np.isclose(annotated.global_phase, np.pi / 4)
assert annotated.metadata["experiment"] == "demo"

annotated.name = "renamed"
assert annotated.name == "renamed" == annotated.circuit.name  # kept in sync with the encoded circuit
print("Circuit metadata round-trips through the encoded circuit ✔")

Circuit metadata round-trips through the encoded circuit ✔


## 2. Addressing qudits and cldigits

Every method that targets qudits/cldigits accepts a flexible "specifier": a single index, a `Qudit`/`ClDigit` object, a `QuditRegister`/`ClDigitRegister`, a `slice`, or a sequence of any of those. Broadcasting is automatic.

In [8]:
addr = QuditQuantumCircuit(4, dim=3, name="addressing_demo")
addr.h(addr.qdregs[0])                 # a whole register -> broadcasts to every member
assert addr.size() == 4

by_list = QuditQuantumCircuit(4, dim=3)
by_list.x([0, 2])                      # a list of indices
assert by_list.size() == 2

by_qudit = QuditQuantumCircuit(4, dim=3)
by_qudit.x(by_qudit.qudits[3])         # a concrete `Qudit` object
assert by_qudit.size() == 1

by_slice = QuditQuantumCircuit(4, dim=3)
by_slice.x(slice(0, 3))                # a slice -> qudits 0, 1, 2
assert by_slice.size() == 3

print("All qudit specifier forms resolve as documented ✔")

All qudit specifier forms resolve as documented ✔


## 3. The single-qudit gate library

`i`, `h`/`hdg`, `k`, `not_`/`qnot`, `x`/`xdg`, `z`/`zdg`, `s`/`sdg`, `t`/`tdg` are parameter-free; `p(theta, qudit)` takes an angle (angle first, mirroring `QuantumCircuit.p`). Every one of them is instantiated *per target qudit*, using that qudit's own dimension — this is what makes mixed-dimension circuits work transparently.

In [9]:
single_qudit_gates = (
    "i", "h", "hdg", "k", "not_", "qnot",
    "x", "xdg", "z", "zdg", "s", "sdg", "t", "tdg",
)

for gate_name in single_qudit_gates:
    trial = QuditQuantumCircuit(1, dim=5)      # a 5-level qudit
    getattr(trial, gate_name)(0)
    assert trial.size() == 1
    assert trial.count_ops()[trial.data[0].name] == 1

phase_trial = QuditQuantumCircuit(1, dim=5)
phase_trial.p(np.pi / 5, 0)                    # angle-first, like `QuantumCircuit.p`
assert phase_trial.size() == 1

print(f"{len(single_qudit_gates)} single-qudit gate helpers + p() all apply cleanly on a 5-level qudit.")

14 single-qudit gate helpers + p() all apply cleanly on a 5-level qudit.


### Stability check: every gate cancels its own adjoint

We build a non-trivial state (`h(0)`), apply a gate followed by its documented adjoint, and check with `state_fidelity` that we land back exactly on the un-touched reference state. This also exercises `copy()`.

In [10]:
def assert_cancels(build_forward, build_inverse, dim=4, atol=1e-7):
    """Apply `forward` then `inverse` on top of a superposed qudit and check
    that it lands back exactly where the untouched reference circuit is."""
    reference = QuditQuantumCircuit(1, dim=dim)
    reference.h(0)                       # a non-trivial superposition

    trial = reference.copy()             # independent copy, exercises `copy()`
    build_forward(trial)
    build_inverse(trial)

    fidelity = state_fidelity(Statevector(reference.circuit), Statevector(trial.circuit))
    assert np.isclose(fidelity, 1.0, atol=atol), fidelity
    return fidelity


adjoint_pairs = [
    (lambda c: c.h(0), lambda c: c.hdg(0)),
    (lambda c: c.s(0), lambda c: c.sdg(0)),
    (lambda c: c.t(0), lambda c: c.tdg(0)),
    (lambda c: c.x(0), lambda c: c.xdg(0)),
    (lambda c: c.z(0), lambda c: c.zdg(0)),
    (lambda c: c.p(0.9, 0), lambda c: c.p(-0.9, 0)),
]

for forward, inverse in adjoint_pairs:
    assert_cancels(forward, inverse)

print("Every gate / adjoint pair restores the original state (fidelity = 1) ✔")

Every gate / adjoint pair restores the original state (fidelity = 1) ✔


## 4. State preparation

`initialize_levels` sets basis states **by level** (never by a concatenated bit-string, which would be ambiguous beyond qubits). A whitespace-separated string is read Qiskit-style: the *rightmost* token is the *first* target qudit.

In [11]:
order_demo = QuditQuantumCircuit(2, dim=16, name="ordering_demo")
order_demo.initialize_levels("11 3")     # rightmost token '3' -> qudit 0, '11' -> qudit 1
order_demo.measure_all(add_digits=True)

raw_counts, decoded = sample_counts(order_demo, shots=200)
assert len(decoded) == 1                 # a basis state is measured deterministically
(observed_levels,) = decoded.keys()
assert observed_levels == (3, 11)

print("initialize_levels('11 3') -> qudit0=3, qudit1=11, exactly as documented ✔")

initialize_levels('11 3') -> qudit0=3, qudit1=11, exactly as documented ✔


In [12]:
# Equivalent, using a plain sequence of levels in *target order* (not reversed):
sequence_demo = QuditQuantumCircuit(2, dim=16, name="ordering_demo_seq")
sequence_demo.initialize_levels([3, 11])
sequence_demo.measure_all(add_digits=True)

_, decoded_seq = sample_counts(sequence_demo, shots=200)
assert set(decoded_seq) == {(3, 11)}
print("initialize_levels([3, 11]) agrees with the string form ✔")

initialize_levels([3, 11]) agrees with the string form ✔


`initialize` dispatches on its argument type: strings/integers are forwarded to `initialize_levels`, sequences/arrays are treated as amplitudes over the logical qudit space (first target = least significant, like a Qiskit `Statevector`).

In [13]:
amp_demo = QuditQuantumCircuit(1, 1, dim=3, name="arbitrary_state")
amplitudes = np.array([1, 1j, -1]) / np.sqrt(3)     # equal populations, different phases
amp_demo.initialize(amplitudes)
amp_demo.measure(0, 0)

_, decoded = sample_counts(amp_demo, shots=9000)
show_decoded(decoded)

total = sum(decoded.values())
for (level,), shots in decoded.items():
    assert abs(shots / total - 1 / 3) < 0.05, (level, shots / total)

print("initialize() with an amplitude vector reproduces |amplitude|^2 populations ✔")

  levels=0          shots= 2959  (32.88%)
  levels=1          shots= 3052  (33.91%)
  levels=2          shots= 2989  (33.21%)
initialize() with an amplitude vector reproduces |amplitude|^2 populations ✔


## 5. Controlled and multi-qudit gates

`sumx` is the qudit generalisation of `cx`: the target is shifted by the sum of the control levels, modulo the target's own dimension. It broadcasts naturally over multiple controls and/or multiple targets.

In [14]:
# A qudit "Bell pair": h() puts the control in a uniform superposition,
# sumx() then perfectly correlates the target with it.
bell = QuditQuantumCircuit(2, 2, dim=3, name="qudit_bell_pair")
bell.h(0)
bell.sumx(0, 1)          # target (initially |0>) becomes a copy of the control
bell.measure([0, 1], [0, 1])

_, decoded = sample_counts(bell, shots=8000)
show_decoded(decoded)

for (control_level, target_level), shots in decoded.items():
    assert control_level == target_level

print("Every shot satisfies control == target: a qudit 'Bell pair' ✔")

  levels=0 0        shots= 2625  (32.81%)
  levels=1 1        shots= 2746  (34.33%)
  levels=2 2        shots= 2629  (32.86%)
Every shot satisfies control == target: a qudit 'Bell pair' ✔


In [15]:
# Multiple controls sharing one target ...
multi = QuditQuantumCircuit(3, dim=3, name="multi_control")
(instruction,) = multi.sumx([0, 1], 2)
assert instruction.qudits == tuple(multi.qudits)     # (control0, control1, target) order

# ... and one control broadcast over multiple targets.
broadcast = QuditQuantumCircuit(3, dim=3, name="broadcast_target")
instructions = broadcast.sumx(0, [1, 2])
assert len(instructions) == 2
assert broadcast.size() == 2

print("Controlled-gate broadcasting over multiple controls/targets works as documented ✔")

Controlled-gate broadcasting over multiple controls/targets works as documented ✔


In [16]:
swap_demo = QuditQuantumCircuit(2, dim=4, name="swap_demo")
swap_demo.initialize_levels("0 3")   # rightmost '3' -> qudit0, '0' -> qudit1
swap_demo.swap(0, 1)
swap_demo.measure_all(add_digits=True)

_, decoded = sample_counts(swap_demo, shots=200)
assert set(decoded) == {(0, 3)}      # the two levels have been exchanged
print("swap() exchanges qudit states ✔ ->", next(iter(decoded)))

swap() exchanges qudit states ✔ -> (0, 3)


## 6. The quantum Fourier transform

`qft()` acts on the full `d**n`-dimensional subspace of its targets (all sharing the same dimension). We check unitarity directly: `QFT† · QFT = I`, combining `compose`, `inverse`, and `copy`.

In [17]:
prep = QuditQuantumCircuit(2, dim=3, name="qft_prep")
prep.initialize_levels("1 2")            # qudit0 = 2, qudit1 = 1

transform = QuditQuantumCircuit(2, dim=3, name="qft_transform")
transform.qft()                          # QFT over the whole 3x3 = 9 dim space

roundtrip = prep.copy()
roundtrip.compose(transform, inplace=True)
roundtrip.compose(transform.inverse(), inplace=True)

fidelity = state_fidelity(Statevector(prep.circuit), Statevector(roundtrip.circuit))
assert np.isclose(fidelity, 1.0, atol=1e-7)
print(f"QFT† · QFT = Identity  (fidelity = {fidelity:.6f}) ✔")

QFT† · QFT = Identity  (fidelity = 1.000000) ✔


In [18]:
# qft() requires its targets to share one dimension:
try:
    mismatched = QuditQuantumCircuit(QuditRegister(1, 3, "a"), QuditRegister(1, 4, "b"))
    mismatched.qft()
except QuditCircuitError as exc:
    print("Expected error for mismatched dimensions:", exc)

Expected error for mismatched dimensions: "'QFT' needs qudits of equal dimension, got (3, 4)."


## 7. Barriers, reset and measurement

In [19]:
reset_demo = QuditQuantumCircuit(1, 1, dim=3, name="reset_demo")
reset_demo.h(0)          # scramble the qudit first ...
reset_demo.reset(0)      # ... reset() always forces it back to |0>
reset_demo.measure(0, 0)

_, decoded = sample_counts(reset_demo, shots=2000)
assert set(decoded) == {(0,)}
print("reset() deterministically returns the qudit to |0> ✔")

reset() deterministically returns the qudit to |0> ✔


In [20]:
convenience = QuditQuantumCircuit(3, dim=3, name="measure_all_demo")
convenience.h(0)
convenience.sumx(0, 1)
convenience.sumx(0, 2)
convenience.measure_all()          # adds a barrier + a fresh 'meas' ClDigitRegister

assert convenience.cbregs[-1].name == "meas"
assert convenience.num_cldigits == 3
print(convenience.draw(output="text"))

        ┌────────┐┌──────────────┐┌──────────────┐ ░ ┌─┐      
  qd_0: ┤ H(d=3) ├┤0             ├┤0             ├─░─┤M├──────
        └────────┘│  SUMX(d=3,3) ││              │ ░ └╥┘┌─┐   
  qd_1: ──────────┤1             ├┤  SUMX(d=3,3) ├─░──╫─┤M├───
                  └──────────────┘│              │ ░  ║ └╥┘┌─┐
  qd_2: ──────────────────────────┤1             ├─░──╫──╫─┤M├
                                  └──────────────┘ ░  ║  ║ └╥┘
meas: 3/══════════════════════════════════════════════╩══╩══╩═
                                                      0  1  2 


## 8. Circuit views: ideal, real and decomposed

The **ideal** view renders one wire per qudit (great for reading intent); the **real** view renders the encoded circuit (what actually runs); **decomposed** unrolls one level further.

In [21]:
views_demo = QuditQuantumCircuit(2, 2, dim=3, name="views_demo")
views_demo.h(0)
views_demo.sumx(0, 1)
views_demo.measure([0, 1], [0, 1])

print("### ideal view (default) ###")
print(views_demo.draw(output="text"))

print("\n### real view (encoded qubits) ###")
print(views_demo.draw(output="text", view="real"))

### ideal view (default) ###
      ┌────────┐┌──────────────┐┌─┐   
qd_0: ┤ H(d=3) ├┤0             ├┤M├───
      └────────┘│  SUMX(d=3,3) │└╥┘┌─┐
qd_1: ──────────┤1             ├─╫─┤M├
                └──────────────┘ ║ └╥┘
cb: 2/═══════════════════════════╩══╩═
                                 0  1 

### real view (encoded qubits) ###
      ┌────┐      ┌─┐         
qd_0: ┤0   ├──■───┤M├─────────
      │  H │  │   └╥┘┌─┐      
qd_1: ┤1   ├──■────╫─┤M├──────
      └────┘┌─┴──┐ ║ └╥┘┌─┐   
qd_2: ──────┤0   ├─╫──╫─┤M├───
            │  X │ ║  ║ └╥┘┌─┐
qd_3: ──────┤1   ├─╫──╫──╫─┤M├
            └────┘ ║  ║  ║ └╥┘
cb: 4/═════════════╩══╩══╩══╩═
                   0  1  2  3 


In [22]:
ideal_circuit = views_demo.to_ideal_circuit()
encoded_copy = views_demo.to_qubit_circuit()
encoded_live = views_demo.to_qubit_circuit(copy=False)

assert isinstance(ideal_circuit, QuantumCircuit)
assert ideal_circuit.num_qubits == views_demo.num_qudits    # one wire per qudit
assert encoded_copy.num_qubits == views_demo.num_qubits     # one wire per encoding qubit
assert encoded_copy is not views_demo.circuit                # independent copy by default
assert encoded_live is views_demo.circuit                    # live reference when copy=False

print("ideal wires:", ideal_circuit.num_qubits, "| encoded qubits:", views_demo.num_qubits)

ideal wires: 2 | encoded qubits: 4


## 9. Circuit metrics

`size()`/`depth()` exclude *directives* (barriers) by default; `count_ops()` counts everything.

In [23]:
metrics_demo = QuditQuantumCircuit(3, 3, dim=3, name="metrics_demo")
metrics_demo.h(0)
metrics_demo.sumx(0, 1)
metrics_demo.sumx(1, 2)
metrics_demo.barrier()
metrics_demo.measure([0, 1, 2], [0, 1, 2])

print("size :", metrics_demo.size())
print("depth:", metrics_demo.depth())
print("width:", metrics_demo.width())
print("ops  :", metrics_demo.count_ops())

assert metrics_demo.width() == metrics_demo.num_qudits + metrics_demo.num_cldigits == 6
assert metrics_demo.size() == 6            # h + 2*sumx + 3*measure (the barrier is excluded)
assert metrics_demo.depth() == 4           # the barrier still synchronises, but adds no depth
assert metrics_demo.count_ops()["measure"] == 3
assert metrics_demo.count_ops()["barrier"] == 1   # count_ops() does NOT filter directives

size : 6
depth: 4
width: 6
ops  : OrderedDict({'measure': 3, 'SUMX': 2, 'H': 1, 'barrier': 1})


## 10. Decoding measurement results

`decode_counts`/`decode_bitstring` turn raw bit-strings back into tuples of qudit levels, using the circuit's own cldigit layout.

In [24]:
decode_demo = QuditQuantumCircuit(1, 1, dim=5, name="decode_bitstring_demo")
decode_demo.initialize_levels(4)
decode_demo.measure(0, 0)

raw_counts, _ = sample_counts(decode_demo, shots=10)
raw_key = next(iter(raw_counts))
print("raw counts key:", repr(raw_key))

decoded_single = decode_demo.decode_bitstring(raw_key)
assert decoded_single == (4,)
print("decode_bitstring(...) ->", decoded_single)

raw counts key: '100'
decode_bitstring(...) -> (4,)


## 11. Error handling: `QuditCircuitError`

The API validates operand counts, dimensions and register names eagerly, instead of failing deep inside the transpiler.

In [25]:
# 1. Two registers cannot share a name.
try:
    QuditQuantumCircuit(QuditRegister(1, 3, "qd"), QuditRegister(1, 3, "qd"))
except QuditCircuitError as exc:
    print("[register name clash]   ", exc)

# 2. A qudit cannot be both a control and the target of the same gate.
dup_err = QuditQuantumCircuit(2, dim=3)
try:
    dup_err.sumx(0, 0)
except QuditCircuitError as exc:
    print("[duplicate operand]     ", exc)

# 3. `swap` needs matching operand counts.
swap_err = QuditQuantumCircuit(3, dim=3)
try:
    swap_err.swap([0], [1, 2])
except QuditCircuitError as exc:
    print("[swap count mismatch]   ", exc)

# 4. `measure` needs one cldigit per qudit.
measure_err = QuditQuantumCircuit(2, 1, dim=3)
try:
    measure_err.measure([0, 1], [0])
except QuditCircuitError as exc:
    print("[measure count mismatch]", exc)

print("\nAll guardrails raise QuditCircuitError as documented ✔")

[register name clash]    "register name 'qd' already exists in this circuit."
[duplicate operand]      'duplicate qudit arguments.'
[swap count mismatch]    'swap needs one qudit per qudit, got 1 and 2 qudit(s).'
[measure count mismatch] 'measure needs one cldigit per qudit, got 2 qudit(s) and 1 cldigit(s).'

All guardrails raise QuditCircuitError as documented ✔


## 12. Putting it all together

The exact example from the class docstring: initialize a qutrit pair, put the control in superposition, entangle with `sumx`, and measure.

In [26]:
showcase = QuditQuantumCircuit(2, 2, dim=3, name="docstring_showcase")
showcase.initialize_levels("2 0")     # qudit 0 -> |0>, qudit 1 -> |2>
showcase.h(0)
showcase.sumx(0, 1)                   # qudit CX: |j, k> -> |j, k+j>
showcase.measure([0, 1], [0, 1])

print(showcase.draw())                # ideal view

_, decoded = sample_counts(showcase, shots=6000)
show_decoded(decoded)

for (control_level, target_level), shots in decoded.items():
    assert (target_level - control_level) % 3 == 2      # target started 2 ahead of control

print("\n(target - control) mod 3 == 2 for every shot, matching sumx's arithmetic ✔")

      ┌──────────────────────────┐┌────────┐┌──────────────┐┌─┐   
qd_0: ┤0                         ├┤ H(d=3) ├┤0             ├┤M├───
      │  initialize(0, 2, d=3,3) │└────────┘│  SUMX(d=3,3) │└╥┘┌─┐
qd_1: ┤1                         ├──────────┤1             ├─╫─┤M├
      └──────────────────────────┘          └──────────────┘ ║ └╥┘
cb: 2/═══════════════════════════════════════════════════════╩══╩═
                                                             0  1 
  levels=2 0        shots= 1989  (33.15%)
  levels=0 1        shots= 2023  (33.72%)
  levels=1 2        shots= 1988  (33.13%)

(target - control) mod 3 == 2 for every shot, matching sumx's arithmetic ✔


## Wrap-up

This tour covered circuit construction (integer & register forms, mixed dimensions), qudit/cldigit addressing, the full single-qudit gate set with adjoint-cancellation checks, state preparation (`initialize_levels`/`initialize`), controlled and multi-qudit gates (`sumx`, `swap`, `qft`) with unitarity/correlation checks, directives (`barrier`/`reset`/`measure`/`measure_all`), the ideal/real circuit views, structural operations (`copy`/`compose`/`inverse`), circuit metrics, result decoding, and the `QuditCircuitError` guardrails.